# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates step-by-step exploration of the FAIR² Mental Health Survey dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published on: {metadata.datePublished}")
print(f"Authors: {[a['@id'] for a in metadata.author] if hasattr(metadata, 'author') else 'N/A'}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Discover record sets
record_sets_metadata = dataset.metadata.recordSet
if not record_sets_metadata or len(record_sets_metadata) == 0:
    print("No record sets found in metadata. Attempting to auto-discover...")
    # mlcroissant can auto-discover record sets from the schema
    record_sets_ids = dataset.list_record_sets()
else:
    record_sets_ids = [rs['@id'] for rs in record_sets_metadata]

print("Available Record Sets (by @id):")
for rsid in record_sets_ids:
    print(f"- {rsid}")

print("\nFields in first record set:")
fields = dataset.list_fields(record_set=record_sets_ids[0])
for field in fields:
    print(f"{field['@id']} - {field.get('name','(No name)')} : {field.get('description','')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set by @id
record_sets = record_sets_ids
dataframes = {}

for record_set in record_sets:
    records = list(dataset.records(record_set=record_set))
    df = pd.DataFrame(records)
    dataframes[record_set] = df

# Display columns and preview for primary record set
primary_record_set_id = record_sets[0]
print(f"Columns in '{primary_record_set_id}':")
print(dataframes[primary_record_set_id].columns.tolist())
print("\nFirst 5 records:")
dataframes[primary_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, and grouping data by key attributes.

In [ ]:
# Identify numeric fields for analysis
df = dataframes[primary_record_set_id]

numeric_fields = [col for col in df.columns if df[col].dtype in ['int64', 'float64'] or (df[col].dtype == 'object' and df[col].str.match(r'^\d+$').any())]
print("Numeric fields detected:", numeric_fields)

# For demonstration, pick GAD-7 total score if present, else default to first numeric
gad7_field_id = next((col for col in numeric_fields if 'gad7' in col.lower() or 'gad_7' in col.lower()), None)
numeric_field = gad7_field_id or numeric_fields[0]
print(f"Using numeric field for EDA: {numeric_field}")

# Define a threshold
threshold = 5
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a demographic field if available (e.g., gender or age_category)
demographic_fields = [col for col in df.columns if any(x in col.lower() for x in ['gender', 'sex', 'age', 'education', 'village'])]
group_field = demographic_fields[0] if demographic_fields else numeric_field
if group_field in df.columns and df[group_field].dtype == 'object':
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nGrouped mean {numeric_field} by {group_field}:\n", grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of GAD-7/selected numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field], bins=15, color='skyblue')
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Boxplot grouped by a demographic field
if group_field in df.columns and df[group_field].dtype == 'object':
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Through this notebook, we explored mental health survey data from Kilifi County, Kenya using the `mlcroissant` library. We:
- Loaded dataset metadata and previewed its structure.
- Identified available record sets and their fields via `@id`.
- Extracted primary records and performed basic exploratory analyses, normalization, and grouping.
- Visualized numeric scores and their distributions across demographic groups.

This approach offers a reproducible AI-ready pathway for understanding survey data in support of community health initiatives. Refer to the dataset schema for full compliance and further metadata exploration.

**Next steps:** You may extend this workflow by modeling risk factors, building predictive models, or aligning data with additional croissant schemas for broader analytic interoperability.